# BTCPredictor2 — GPU Training (1-Day Horizon)

## Before running
1. Enable GPU: **Runtime → Change runtime type → A100 GPU** (recommended)
2. Run **Cell 1** — clones repo, installs packages, configures GPU
3. Run **Cell 2** — verifies the 10 yearly CSVs (downloads any missing ones from GitHub)
4. Run **Cell 3** — trains all three models end-to-end via `run_full_training_pipeline()`
5. Run **Cell 4** — packages and downloads the trained models as a zip

## Training data
All yearly merged CSVs live in `data/yearly_merged/` and are committed to the repo.  
Cell 1 clones them automatically. Cell 2 validates presence and downloads any missing files.

- `{YYYY}_bilstm_merged.csv` — 15-min sampled, m15_* + h1_* features (BiLSTM input)
- `{YYYY}_tft_merged.csv`   — 4h sampled,  h4_* + d1_* features  (TFT input)

## Estimated time on A100 GPU
| Phase | Description | Time |
|---|---|---|
| Phase 1 | TFT ensemble (3 seeds × 30-day / 4h context) | 2–4 hours |
| Phase 2 | BiLSTM ensemble (3 seeds × 48h / 15min context) | 1–2 hours |
| Phase 3 | XGBoost meta-learner (OOF + gate training) | ~5 minutes |

## Architecture summary
- **TFT** (macro): binary sigmoid head, `binary_crossentropy` loss
- **BiLSTM/ACB** (micro): price-volume customized attention, `binary_crossentropy` loss
- **Meta (XGBoost)**: agreement-based routing gate, OOF purge = **72h** (matches MAX_HOLD_MIN)
- Both base models output directly comparable `p ∈ [0,1]` — no quantile mismatch

In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT SETUP
# Clones the repo (always fresh), installs packages, enables GPU.
# ============================================================

GITHUB_URL = 'https://github.com/chefo919/BTCPredictor2.git'

import shutil, os, sys

if os.path.exists('/content/BTCPredictor2'):
    shutil.rmtree('/content/BTCPredictor2')
    print('Removed old clone.')

os.system(f'git clone {GITHUB_URL} /content/BTCPredictor2')
os.system('cd /content/BTCPredictor2 && git log --oneline -3')

REPO = '/content/BTCPredictor2'
sys.path.insert(0, REPO)
os.chdir(REPO)

os.system('pip install -q xgboost joblib ta scikit-learn-intelex numba')

os.makedirs(f'{REPO}/data/yearly_merged', exist_ok=True)
os.makedirs(f'{REPO}/data/yearly_1m',     exist_ok=True)
os.makedirs(f'{REPO}/models/saved',       exist_ok=True)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU: {gpus[0].name}  |  Mixed precision: ON')
else:
    print('WARNING: No GPU — Runtime > Change runtime type > GPU')

import config
from features.engineer import get_feature_groups
groups = get_feature_groups()

print()
print(f'BiLSTM features : {len(groups["bilstm"])}  '
      f'(m15/h1 — {config.SEQ_LEN_BILSTM} steps × 15min = 48h context)')
print(f'TFT    features : {len(groups["tft_dynamic"])}  '
      f'(h4/d1  — {config.SEQ_LEN_TFT} steps × 4h = 30 days context)')
print(f'OOF purge       : TFT={config.PURGE_ROWS_TFT} rows = 72h  |  '
      f'BiLSTM={config.PURGE_ROWS_BILSTM} rows = 72h  (matches MAX_HOLD_MIN)')
print()
print('Done. Run Cell 2 to verify training data.')

In [ ]:
# ============================================================
# CELL 2 — VERIFY DATA FILES
# Checks both yearly_merged (features) and yearly_1m (1m OHLC for labels).
# Downloads any missing files directly from GitHub.
# ============================================================

import os, subprocess

GITHUB_RAW = 'https://raw.githubusercontent.com/chefo919/BTCPredictor2/main'
YEARLY_DIR = '/content/BTCPredictor2/data/yearly_merged'
YEARLY_1M  = '/content/BTCPredictor2/data/yearly_1m'
os.makedirs(YEARLY_DIR, exist_ok=True)
os.makedirs(YEARLY_1M,  exist_ok=True)

YEARS = ['2022', '2023', '2024', '2025', '2026']

MERGED_FILES = (
    [f'{y}_bilstm_merged.csv' for y in YEARS] +
    [f'{y}_tft_merged.csv'    for y in YEARS]
)
M1_FILES = [f'{y}_btc_1m.csv' for y in YEARS]


def _download_if_missing(files, local_dir, remote_subdir):
    missing = [f for f in files if not os.path.exists(os.path.join(local_dir, f))]
    if missing:
        print(f'Downloading {len(missing)} missing file(s) from GitHub ({remote_subdir})...')
        for fname in missing:
            url = f'{GITHUB_RAW}/{remote_subdir}/{fname}'
            dst = os.path.join(local_dir, fname)
            r = subprocess.run(['wget', '-q', '--show-progress', '-O', dst, url])
            if r.returncode != 0 or not os.path.exists(dst) or os.path.getsize(dst) < 1000:
                print(f'  ERROR: failed to download {fname}')
            else:
                print(f'  {fname}  ({os.path.getsize(dst)/1024**2:.1f} MB)')
    else:
        print(f'All {len(files)} {remote_subdir} files already present.')


_download_if_missing(MERGED_FILES, YEARLY_DIR, 'data/yearly_merged')
_download_if_missing(M1_FILES,     YEARLY_1M,  'data/yearly_1m')

print()
bilstm_total = tft_total = m1_total = 0
errors = []

for fname in sorted(MERGED_FILES):
    path = os.path.join(YEARLY_DIR, fname)
    if os.path.exists(path):
        rc = sum(1 for _ in open(path)) - 1
        sz = os.path.getsize(path) / 1024**2
        tag = 'bilstm' if 'bilstm' in fname else 'tft   '
        print(f'  [{tag}] {fname}: {rc:,} rows  ({sz:.1f} MB)')
        if 'bilstm' in fname: bilstm_total += rc
        else:                  tft_total    += rc
    else:
        errors.append(fname); print(f'  MISSING: {fname}')

print()
for fname in sorted(M1_FILES):
    path = os.path.join(YEARLY_1M, fname)
    if os.path.exists(path):
        rc = sum(1 for _ in open(path)) - 1
        sz = os.path.getsize(path) / 1024**2
        print(f'  [1m    ] {fname}: {rc:,} rows  ({sz:.1f} MB)')
        m1_total += rc
    else:
        errors.append(fname); print(f'  MISSING: {fname}')

print()
print(f'Total bilstm rows: {bilstm_total:,}  |  tft rows: {tft_total:,}  |  1m rows: {m1_total:,}')
if errors:
    print(f'WARNING: {len(errors)} file(s) missing — check repo has data/ committed')
else:
    print('All files verified.  Run Cell 3 to start training.')

In [ ]:
# ============================================================
# CELL 3 — FULL TRAINING PIPELINE  (single call)
#
# Runs all three phases sequentially:
#   Phase 1: TFT ensemble   (2–4h on A100)
#   Phase 2: BiLSTM ensemble (1–2h on A100)
#   Phase 3: XGBoost meta-learner (~5 min)
#
# config_override keys used here:
#   batch_tft / batch_bilstm  — large GPU batches for speed
#   force                     — True: retrain even if saved models exist
# ============================================================

import os, sys
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

from training.train import run_full_training_pipeline

results = run_full_training_pipeline(config_override={
    'batch_tft':    512,    # 32 on CPU → 512 on A100
    'batch_bilstm': 512,    # 16 on CPU → 512 on A100
    'force':        True,   # retrain all seeds from scratch
})

print()
print('=' * 50)
print('FINAL RESULTS')
print(f'  TFT accuracy:    {results["tft_acc"]:.4f}')
print(f'  BiLSTM accuracy: {results["bilstm_acc"]:.4f}')
print(f'  Meta routing:    {results["meta_acc"]:.4f}')
print(f'  Gate CV acc:     {results["gate_cv_acc"]:.4f}')
w = results.get('weights', {})
print(f'  Static fallback: TFT={w.get("tft",0):.3f}  BiLSTM={w.get("bilstm",0):.3f}')
print('=' * 50)
print('Run Cell 4 to package and download model files.')

In [ ]:
# ============================================================
# CELL 4 — PACKAGE AND DOWNLOAD MODELS
# Copies all trained model files to a zip for local deployment.
# Extract into your local models/saved/ folder.
# ============================================================

import os, shutil
import config

OUT = '/content/models_output'
os.makedirs(OUT, exist_ok=True)

# Per-seed model files
for s in range(config.N_ENSEMBLE):
    for fname in [f'tft_s{s}.keras',      f'tft_scaler_s{s}.pkl',
                  f'bilstm_s{s}.keras',   f'bilstm_scaler_s{s}.pkl']:
        src = f'models/saved/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{OUT}/{fname}')
        else:
            print(f'  WARNING: {fname} not found — was training completed?')

# Shared model files
for fname in ['meta_xgb.pkl', 'model_accuracies.json', 'training_cutoff.txt']:
    src = f'models/saved/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{fname}')

saved_files = sorted(os.listdir(OUT))
print(f'Packaged {len(saved_files)} files:')
for f in saved_files:
    sz = os.path.getsize(f'{OUT}/{f}') / 1024**2
    print(f'  {f}  ({sz:.1f} MB)')

shutil.make_archive('/content/btc_models', 'zip', OUT)
zip_mb = os.path.getsize('/content/btc_models.zip') / 1024**2
print(f'\nCreated btc_models.zip  ({zip_mb:.1f} MB)')

from google.colab import files
files.download('/content/btc_models.zip')
print('\nExtract the zip and copy all files into your local models/saved/ folder.')
print('Then run: python prediction/predict.py')